# Swedish Red Cross: RAG-assisted application analysis

This notebook demonstrates a small, local proof of concept for the proposed internship use case:

> Retrieve relevant passages from previous applications, classify the application into structured fields, and draft an auditable answer grounded in those passages.

The included documents are synthetic examples. No real applicant or donor data is included.

In [ ]:
from pathlib import Path
import sys

# Make the notebook work from the repository root or from the notebooks folder.
repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from rag_project.classifier import classify_application
from rag_project.loaders import load_markdown_documents
from rag_project.pipeline import RagPipeline

documents_path = repo_root / 'data' / 'red_cross_examples'
print(f'Repository: {repo_root}')
print(f'Documents: {documents_path}')

## 1. Ingest and inspect the source material

In production, this step would include access control, file validation, versioning, and metadata such as country, programme, donor, and approval status.

In [ ]:
documents = load_markdown_documents(documents_path)
print(f'Loaded {len(documents)} Markdown documents')
for document in documents:
    print(f'- {document.source}: {len(document.text.split())} words')

## 2. Build the RAG pipeline

This proof of concept uses TF-IDF retrieval because it is transparent and runs without an external service. The architecture keeps retrieval separate from generation, so embeddings and a vector database can be introduced later.

In [ ]:
pipeline = RagPipeline.from_documents(
    documents_path,
    chunk_size=100,
    overlap=20,
)
print(f'Indexed {len(pipeline.chunks)} chunks')

In [ ]:
question = 'How can previous applications be classified to support future reports?'
answer = pipeline.ask(question, top_k=3)

print('ANSWER')
print(answer.answer)
print('\nSOURCES')
for source in answer.citations:
    print(f'- {source}')

## 3. Classify one application into structured metadata

The structured output is intended for a searchable database and later Power BI reporting. A human reviewer remains responsible for approving the classification.

In [ ]:
application = '''
Programme Area: Community resilience
Country: Kenya
Donor Type: Institutional donor
Target Group: Young adults and community volunteers
Expected outcomes: improved preparedness, stronger referral pathways
The project will track indicators related to trained volunteers, referral completion and households reached.
'''

classification = classify_application(application)
print(f'Program area: {classification.program_area}')
print(f'Geography: {classification.geography}')
print(f'Donor type: {classification.donor_type}')
print(f'Target group: {classification.target_group}')
print(f'Outcomes: {classification.outcomes}')
print(f'Indicators: {classification.indicators}')
print(f'Initial confidence: {classification.confidence:.0%}')

## 4. A small evaluation check

Before using this with real applications, I would create an annotated test set with domain experts and measure field-level accuracy, retrieval precision@k, citation coverage, and human correction rate. This cell illustrates the idea with a simple expected-source check.

In [ ]:
evaluation_questions = [
    ('What fields are useful for classifying previous applications?', 'application_classification.md'),
    ('How can retrieved content support report drafting?', 'report_drafting.md'),
]

hits = 0
for query, expected_source in evaluation_questions:
    result = pipeline.ask(query, top_k=3)
    found = any(expected_source in source for source in result.citations)
    hits += int(found)
    print(f'{found!s:5} | {query}')

print(f'Expected-source hit rate: {hits / len(evaluation_questions):.0%}')

## 5. Production evolution

- Replace TF-IDF with multilingual embeddings and a vector database.
- Add document-level permissions and remove or mask personal data.
- Require human review for low-confidence or high-impact classifications.
- Store source citations and model/version metadata for auditability.
- Connect approved structured data to Power BI and use retrieval only from approved sources.

The central design principle is assistive AI: the system accelerates review and drafting, while people retain decision authority.